In [1]:
print("hello world")

hello world


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY1"]=os.getenv("OPENAI_API_KEY1")
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [3]:
from langchain_community.document_loaders import WebBaseLoader
web_loader = WebBaseLoader("https://docs.langchain.com/oss/python/langchain/models")


c:\Users\asrit\Videos\langchain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
web_loader.load()

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/models', 'title': 'Models - Docs by LangChain', 'language': 'en'}, page_content='Models - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationCore componentsModelsOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext eng

In [7]:
docs=web_loader.load()

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

In [12]:
docs=splitter.split_documents(docs)

In [13]:
docs

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/models', 'title': 'Models - Docs by LangChain', 'language': 'en'}, page_content="Models - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationCore componentsModelsOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engi

In [15]:
from langchain_ollama import OllamaEmbeddings
embed=OllamaEmbeddings(model="nomic-embed-text")

In [17]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(docs,embed)

In [20]:
db=vectorstoredb.as_retriever()

In [23]:
query="lang smith has two languages"

vector=vectorstoredb.similarity_search(query)
print(vector)

[Document(id='7e845eca-188c-4c19-b1bb-5733ccfcffa1', metadata={'source': 'https://docs.langchain.com/oss/python/langchain/models', 'title': 'Models - Docs by LangChain', 'language': 'en'}, page_content='For provider-specific integration information and capabilities, see the provider’s chat model page.\nLangSmith traces each model call so you can compare providers, inspect tool routing, and debug failures. Follow the tracing quickstart to get set up.We recommend you also set up LangSmith Engine which monitors your traces, detects issues, and proposes fixes.\n\u200bBasic usage\nModels can be utilized in two ways:'), Document(id='01a5ad02-77dc-4bb3-8702-1e14de967f82', metadata={'source': 'https://docs.langchain.com/oss/python/langchain/models', 'title': 'Models - Docs by LangChain', 'language': 'en'}, page_content='See init_chat_model for more detail, including information on how to pass model parameters.\n\u200bSupported providers and models\nLangChain supports all major model providers 

In [25]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based on the provied context
<context>
{context}
</context>
    """
)

In [26]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI()


In [28]:
documentchain=create_stuff_documents_chain(llm,prompt)
documentchain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based on the provied context\n<context>\n{context}\n</context>\n    '), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openai': '1.4.3'}}, output_version=None, profile={'name': 'GPT-3.5-turbo', 'status': 'deprecated', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': Fa

In [29]:
from langchain_core.documents import Document
documentchain.invoke(
    {
        "input":"LLMs are powerful AI tools that can interpr",
        "context":[Document(page_content="LLMs are powerful AI tools that can interpret and generate text like humans. They’re versatile enough to write content, translate languages, summarize, and answer questions without needing specialized training for each task ")]

    }
)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [30]:
vectorstoredb

In [32]:
new_db=vectorstoredb.as_retriever()
new_db.invoke("hi")

[Document(id='d360610f-221e-4716-b701-b4cf5ba3ab6a', metadata={'source': 'https://docs.langchain.com/oss/python/langchain/models', 'title': 'Models - Docs by LangChain', 'language': 'en'}, page_content='else:\n        pass\nInput: Hello\nToken: Hi\nToken:  there\nToken: !\nToken:  How\nToken:  can\nToken:  I\n...\nFull message: Hi there! How can I help today?'),
 Document(id='d3313368-117c-4ddf-a7c8-c7218e1a9cf5', metadata={'source': 'https://docs.langchain.com/oss/python/langchain/models', 'title': 'Models - Docs by LangChain', 'language': 'en'}, page_content='first_model.invoke("what\'s your name")\nfirst_model.invoke(\n    "what\'s your name",\n    config={\n        "configurable": {\n            "first_model": "claude-sonnet-4-6",\n            "first_temperature": 0.5,\n            "first_max_tokens": 100,\n        }\n    },\n)\nSee the init_chat_model reference for more details on configurable_fields and config_prefix.\nUsing a configurable model declarativelyWe can call declarati

In [34]:
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(new_db,documentchain)

In [35]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000197D5780B90>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based on the provied context\n<context>\n{context}\n</context>\n    '), additional_kwargs={})])
            | ChatOpe

In [36]:
response=retrieval_chain.invoke({"input":"LLMs are powerful AI tools that can interpr"})
response['answer']

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}